# 🏗️ System Design Whiteboard — The Master Guide
### *From Zero to Interview-Ready*

---

> **Mental Model First:**
> A system design interview is like being asked to design a city's water system in 45 minutes. You don't know yet if it's for a town of 10,000 or a metropolis of 10 million. Before drawing any pipes, you ask: how many people, what's the usage pattern, what breaks first? Then you sketch the big zones, identify the one pipe that everything depends on, and describe how you'd scale each zone independently.

---

## 📋 Table of Contents

| # | Section |
|---|--------|
| 1 | [The Whiteboard Framework](#1) |
| 2 | [Interview Structure & Timing](#2) |
| 3 | [Core Vocabulary & Trade-offs](#3) |
| 4 | [Decision Map — Component Selection](#4) |
| 5 | [Pattern 1: Batch Data Pipeline Design](#5) |
| 6 | [Pattern 2: Streaming Pipeline Design](#6) |
| 7 | [Pattern 3: Lambda / Kappa Architecture](#7) |
| 8 | [Pattern 4: AWS Stack — End-to-End](#8) |
| 9 | [Pattern 5: Trade-off Reasoning & SLAs](#9) |
| 10 | [Full Decision Map](#10) |
| 11 | [Interview Cheat Sheet](#11) |
| 12 | [Summary Map](#12) |

<a id='1'></a>
## 1. The Whiteboard Framework

```
THE 6-STEP SYSTEM DESIGN FRAMEWORK
──────────────────────────────────────────────────────────────────

  STEP 1 — CLARIFY (5 min)
  ─────────────────────────
  Ask before drawing anything:
    - What is the data source? (events, DB, files, APIs)
    - What is the consumer? (dashboards, ML, analysts, API)
    - What is the volume? (rows/sec, GB/day)
    - What is the latency requirement? (real-time, 5-min, hourly, daily)
    - What is the SLA? (uptime, data freshness, correctness guarantee)
    - Any existing stack I should build on or avoid?

  STEP 2 — HIGH-LEVEL SKETCH (5 min)
  ────────────────────────────────────
  Draw the three zones: INGEST → STORE → SERVE
  Name each box; don't optimize yet.

     [Source] → [Ingest] → [Storage] → [Processing] → [Serving]

  STEP 3 — DEEP-DIVE ONE COMPONENT (10 min)
  ───────────────────────────────────────────
  Interviewer picks OR you pick the hardest / most interesting.
  Show you can go deep: schema, partitioning, retry logic, failure modes.

  STEP 4 — SCALE IT (10 min)
  ───────────────────────────
  "What if this is 100x more data?"
  Identify the bottleneck; explain how you'd scale each layer.

  STEP 5 — RELIABILITY & FAILURE (5 min)
  ────────────────────────────────────────
  What breaks? What's the blast radius? How do you detect and recover?

  STEP 6 — TRADE-OFF SUMMARY (5 min)
  ─────────────────────────────────────
  "Given constraints X, I chose Y over Z because..."
  Show you know what you gave up.
```

<a id='2'></a>
## 2. Interview Structure & Timing

In [ ]:
# Interview timing model — how to allocate 45-minute system design

interview_phases = [
    {"phase": "Clarify requirements",      "minutes": 5,  "what_to_show": "You don't over-engineer; you ask the right questions"},
    {"phase": "High-level sketch",          "minutes": 7,  "what_to_show": "You can see the full system before diving into details"},
    {"phase": "Deep-dive: storage / schema","minutes": 10, "what_to_show": "Technical depth; partitioning, schema, trade-offs"},
    {"phase": "Deep-dive: processing",      "minutes": 8,  "what_to_show": "Batch vs stream, idempotency, failure handling"},
    {"phase": "Scale discussion",           "minutes": 7,  "what_to_show": "Back-of-envelope math; identifying bottlenecks"},
    {"phase": "Reliability & monitoring",   "minutes": 5,  "what_to_show": "SLOs, alerting, DLQ, data quality checks"},
    {"phase": "Trade-off summary",          "minutes": 3,  "what_to_show": "Judgment: what you chose and what you gave up"},
]

print("SYSTEM DESIGN INTERVIEW — TIME ALLOCATION (45 min)")
print("=" * 60)
elapsed = 0
for p in interview_phases:
    end = elapsed + p["minutes"]
    print(f"  {elapsed:>2}-{end:<2} min  [{p['phase']:<32}]")
    print(f"            Show: {p['what_to_show']}")
    elapsed = end

print(f"\n  Total: {elapsed} minutes")

# Key clarifying questions — always ask these first
print("\n\nCLARIFYING QUESTIONS (ask before drawing anything)")
print("-" * 55)
questions = [
    ("Volume",       "How many events/rows per second at peak? What's the daily GB?"),
    ("Latency",      "Real-time (<1s)? Near-real-time (<5min)? Batch (hourly/daily)?"),
    ("Consumers",    "Who reads this data? Dashboard, ML model, API, analyst?"),
    ("Correctness",  "At-least-once OK, or exactly-once required?"),
    ("Retention",    "How long must we keep data? 30 days? 7 years (compliance)?"),
    ("SLA",          "What's the uptime requirement? Allowed downtime per month?"),
    ("Existing stack","Greenfield or must I use existing AWS/GCP/Snowflake?"),
    ("Budget",       "Cost sensitivity? Or scale-at-any-cost startup mode?"),
]
for topic, q in questions:
    print(f"  [{topic:<15}] {q}")

<a id='3'></a>
## 3. Core Vocabulary & Trade-offs

```
KEY TRADE-OFFS TO ALWAYS MENTION
──────────────────────────────────────────────────────────────────
TRADE-OFF            OPTION A              OPTION B
────────────────────────────────────────────────────────────────
Delivery semantics   At-least-once         Exactly-once
                     (simpler, fast)       (expensive, complex)

Consistency          Strong consistency    Eventual consistency
                     (slow, expensive)     (fast, cheap, possible staleness)

Latency vs cost      Real-time (streaming) Batch (cheaper, higher latency)

Schema               Schema-on-write       Schema-on-read
                     (enforced, rigid)     (flexible, risk of bad data)

Storage format       Row (fast writes)     Columnar (fast scans)
                     (OLTP, low latency)   (OLAP, analytics)

Fan-out              One stream, many      Many streams, one per consumer
                     consumers             (isolated, easier ops)

BACK-OF-ENVELOPE NUMBERS (memorize these)
──────────────────────────────────────────────────────────────────
  1M events/day   = ~12 events/sec
  1B events/day   = ~12k events/sec
  1 KB/event × 1M/day = 1 GB/day
  1 KB/event × 1B/day = 1 TB/day
  Parquet compression: ~5-10x vs raw JSON
  S3 cost: ~$0.023/GB/month
  Kinesis: 1 shard = 1 MB/s write, 2 MB/s read, 1000 records/s
  Lambda: max 15 min, 10GB RAM, 1000 concurrent default
  DynamoDB: 1 WCU = 1 write/s up to 1KB
  Redshift: 1 node = ~160GB compressed, ~2 TB raw
```

In [ ]:
# Back-of-envelope calculator — use this in interviews

def size_pipeline(
    events_per_day: int,
    bytes_per_event: int,
    retention_days: int = 365,
    parquet_compression: float = 7.0,
    s3_cost_per_gb_month: float = 0.023,
):
    """
    Back-of-envelope sizing for a data pipeline.
    Used in system design interviews to justify component choices.
    """
    events_per_sec = events_per_day / 86_400
    raw_gb_per_day = (events_per_day * bytes_per_event) / 1e9
    parquet_gb_per_day = raw_gb_per_day / parquet_compression
    total_parquet_gb = parquet_gb_per_day * retention_days
    monthly_s3_cost = (total_parquet_gb / retention_days * 30) * s3_cost_per_gb_month

    # Kinesis shard sizing: 1 shard = 1 MB/s write = 1000 records/s
    mb_per_sec = (events_per_day * bytes_per_event) / 86_400 / 1e6
    shards_by_throughput = max(1, int(mb_per_sec / 1.0) + 1)
    shards_by_rps = max(1, int(events_per_sec / 1000) + 1)
    kinesis_shards = max(shards_by_throughput, shards_by_rps)

    print("PIPELINE SIZING")
    print("=" * 45)
    print(f"  Events/day:          {events_per_day:>15,}")
    print(f"  Events/sec (avg):    {events_per_sec:>15.1f}")
    print(f"  Bytes/event:         {bytes_per_event:>15,} B")
    print(f"  Raw GB/day:          {raw_gb_per_day:>15.2f} GB")
    print(f"  Parquet GB/day:      {parquet_gb_per_day:>15.2f} GB  (÷{parquet_compression}x)")
    print(f"  Total storage ({retention_days}d): {total_parquet_gb:>15.0f} GB")
    print(f"  S3 cost/month:       ${monthly_s3_cost:>14.2f}")
    print(f"  Kinesis shards:      {kinesis_shards:>15} shards")

    # Component recommendations
    print("\nCOMPONENT RECOMMENDATIONS:")
    if events_per_sec < 100:
        print("  Ingest:    SQS or EventBridge (low volume, no Kinesis needed)")
    elif events_per_sec < 10_000:
        print(f"  Ingest:    Kinesis ({kinesis_shards} shards)")
    else:
        print("  Ingest:    Kinesis + MSK (Kafka) for very high volume")

    if parquet_gb_per_day < 10:
        print("  Process:   Lambda (low volume, no Spark overhead)")
    elif parquet_gb_per_day < 1000:
        print("  Process:   Glue Spark or EMR (medium volume)")
    else:
        print("  Process:   EMR Spark with autoscaling (high volume)")

    print("  Storage:   S3 + Parquet (analytical) + DynamoDB (key lookups)")
    print("  Query:     Athena (ad-hoc) + Redshift (BI dashboards)")


print("SCENARIO 1: Medium SaaS — 10M events/day, 1KB each")
size_pipeline(events_per_day=10_000_000, bytes_per_event=1_000)

print("\n" + "─"*50)
print("SCENARIO 2: Large platform — 1B events/day, 500B each")
size_pipeline(events_per_day=1_000_000_000, bytes_per_event=500)

<a id='4'></a>
## 4. Decision Map — Component Selection

```
REQUIREMENT → COMPONENT SELECTION
──────────────────────────────────────────────────────────────────
Latency requirement:
  <1 second     → Kinesis + Lambda / Flink / Spark Structured Streaming
  1-5 minutes   → Kinesis → Lambda (micro-batch)
  Hourly        → Scheduled Glue / EMR / Airflow DAG
  Daily         → S3 batch + Glue Crawler + Athena

Volume (events/sec):
  <100/sec      → SQS → Lambda (no Kinesis needed)
  100–10k/sec   → Kinesis (N shards) → Lambda consumer
  >10k/sec      → Kinesis + Firehose / MSK (Kafka on EMR)

Query pattern:
  Key-value lookup (<10ms)    → DynamoDB (single-table)
  Ad-hoc SQL on S3            → Athena
  BI dashboards, complex SQL  → Redshift
  Full-text search            → OpenSearch
  Operational SQL, OLTP       → Aurora / RDS

Delivery guarantee:
  At-least-once (acceptable)  → SQS, Kinesis default
  Exactly-once required       → Kafka transactions, Spark + checkpoints,
                                 idempotent writes + dedup key

Schema management:
  Schema must be enforced     → Schema Registry (Glue, Confluent)
  Flexible / evolving schema  → S3 + Parquet + schema evolution (Iceberg)
  Unknown schema at write time → S3 raw + Schema-on-read (Athena)

Cost priority:
  Minimize cost               → S3 + Athena + scheduled Glue
  Minimize latency (cost ++)  → Kinesis + Flink + DynamoDB + ElastiCache
  Managed, less ops           → Firehose + Glue + Athena (serverless stack)
```

<a id='5'></a>
## 5. Pattern 1: Batch Data Pipeline Design

---

```
PROBLEM:
  Design a daily batch pipeline that ingests 10M orders/day from
  an OLTP database, transforms them, and loads to a data warehouse
  for BI reporting. Freshness: T+2 hours. SLA: 99.5% on-time.

ARCHITECTURE:
  [RDS Orders DB]
      │  (CDC via DMS or scheduled JDBC extract)
      ▼
  [S3 Raw Zone]          ← Parquet, partitioned by ingestion_date
      │  (Glue ETL job)
      ▼
  [S3 Curated Zone]      ← cleaned, deduplicated, typed
      │  (Glue ETL or dbt)
      ▼
  [S3 Aggregated Zone]   ← daily order summaries, customer totals
      │  (COPY command)
      ▼
  [Redshift]             ← final DW tables for BI dashboards

SLOW MOTION TRACE — Key design decisions:
  Decision 1: CDC vs full extract
    → CDC (DMS) for high-volume tables (track only deltas)
    → Full extract for small reference/dimension tables

  Decision 2: Partition strategy
    → Partition by ingestion_date (not event_date) in raw zone
    → Partition by event_date in curated zone (query pattern: time range)

  Decision 3: Idempotency
    → Glue job: overwrite S3 partition on each run (DELETE + write)
    → Redshift: MERGE on primary key (upsert, not append)

  Decision 4: Orchestration
    → Airflow DAG: extract → validate → transform → load
    → Data quality check as a gate: fails if null_rate > 1%

KEY INSIGHT:
  Idempotency is the single most important property of a batch pipeline.
  Every step must be safe to re-run. Design for the retry, not the happy path.
```

In [ ]:
# Batch pipeline design simulator — walks through all design decisions

class BatchPipelineDesign:
    """
    Simulates the decisions made when designing a batch DE pipeline.
    Used for whiteboard interview practice — talk through each step.
    """

    def __init__(self, name, events_per_day, latency_target_hours, sla_uptime):
        self.name = name
        self.events_per_day = events_per_day
        self.latency_target_hours = latency_target_hours
        self.sla_uptime = sla_uptime

    def select_ingest_pattern(self):
        # CDC = capture only changed rows; full extract = all rows
        if self.events_per_day > 5_000_000:
            return "CDC via AWS DMS (delta only — too large for full extract)"
        return "Scheduled JDBC full extract (manageable volume)"

    def select_storage_format(self):
        # Parquet + columnar = fast analytical scans
        return "S3 + Parquet (columnar, ~7x compression vs JSON, predicate pushdown)"

    def select_partition_strategy(self):
        # Raw zone: partition by when data arrived (dedup window)
        # Curated zone: partition by event time (query filter)
        return (
            "Raw:     s3://bucket/raw/ingestion_date=YYYY-MM-DD/  (arrived time)\n"
            "Curated: s3://bucket/curated/event_date=YYYY-MM-DD/  (event time)\n"
            "Agg:     s3://bucket/agg/region=X/event_date=YYYY-MM-DD/"
        )

    def select_processing_engine(self):
        gb_per_day = (self.events_per_day * 1000) / 1e9  # assume 1KB/event
        if gb_per_day < 10:
            return "AWS Lambda (small volume, no cluster startup overhead)"
        elif gb_per_day < 1000:
            return "AWS Glue Spark (managed, serverless, good for 10-1000 GB)"
        return "EMR Spark (large volume, need autoscaling + spot instances)"

    def idempotency_strategy(self):
        return [
            "S3 write: delete partition first, then write (overwrite, not append)",
            "Redshift load: MERGE on primary key (not INSERT — avoids duplicates)",
            "Airflow task: set max_active_runs=1 (no concurrent runs of same DAG)",
            "Data quality gate: validate before load — fail DAG, don't load bad data",
        ]

    def failure_modes(self):
        return [
            ("Source DB slow",    "Timeout → retry with backoff; alert if > 3 failures"),
            ("Bad data in source","DQ gate fails → quarantine to S3 bad-data/ → alert"),
            ("Glue job OOM",      "Increase DPU; add repartition() before heavy transforms"),
            ("Redshift COPY fails","Idempotent — retry; check STL_LOAD_ERRORS table"),
            ("DAG misses SLA",    "SLA alarm → page on-call; look at task durations"),
        ]

    def print_design(self):
        print(f"BATCH PIPELINE DESIGN: {self.name}")
        print("=" * 60)
        print(f"  Volume:           {self.events_per_day:,} events/day")
        print(f"  Latency target:   T+{self.latency_target_hours}h")
        print(f"  SLA uptime:       {self.sla_uptime}%")
        print(f"\n[INGEST]     {self.select_ingest_pattern()}")
        print(f"\n[STORAGE]    {self.select_storage_format()}")
        print(f"\n[PARTITION]\n{self.select_partition_strategy()}")
        print(f"\n[PROCESSING] {self.select_processing_engine()}")
        print("\n[IDEMPOTENCY]")
        for s in self.idempotency_strategy():
            print(f"  - {s}")
        print("\n[FAILURE MODES]")
        for mode, response in self.failure_modes():
            print(f"  {mode:<25} → {response}")


design = BatchPipelineDesign(
    name="Daily Orders ETL → Redshift",
    events_per_day=10_000_000,
    latency_target_hours=2,
    sla_uptime=99.5
)
design.print_design()

<a id='6'></a>
## 6. Pattern 2: Streaming Pipeline Design

---

```
PROBLEM:
  Design a real-time pipeline that processes clickstream events from
  a web app (500k events/min peak), detects fraud patterns, and
  surfaces alerts within 30 seconds. Must not lose events.

ARCHITECTURE:
  [Web App]
      │  (SDK → REST API → Kinesis PutRecord)
      ▼
  [Kinesis Data Streams]   ← 9 shards (500k/min = ~8.3k/sec ÷ 1k rec/shard = 9)
      │
      ├─────────────────────────────────────────────┐
      │  (consumer 1)                               │  (consumer 2)
      ▼                                             ▼
  [Lambda: fraud detection]              [Firehose → S3: raw archive]
      │  (sliding window, 30s)
      ▼
  [DynamoDB: alert state]                [Athena: historical analysis]
      │
      ▼
  [SNS → ops team alert]

SLOW MOTION TRACE — Streaming design decisions:

  Decision 1: Kinesis shard count
    500k events/min = 8,333 events/sec
    1 shard = 1,000 records/sec → 9 shards (add buffer)

  Decision 2: Lambda consumer design
    - 1 Lambda per shard (Kinesis triggers)
    - Sliding window: DynamoDB counts events per user per 30s
    - Threshold breach → SNS alert

  Decision 3: Fan-out for archival
    - Firehose taps the same stream separately (no Lambda involvement)
    - Writes to S3 every 60s or 128MB (configurable buffer)

  Decision 4: Exactly-once for fraud detection
    - Use DynamoDB conditional write (only insert if not exists)
    - Dedup key = event_id (UUID from client)
    - At-most-once is acceptable for alerts (false-negative ok)

KEY INSIGHT:
  Fan-out consumers read the same Kinesis stream independently.
  Each consumer has its own iterator position — they don't interfere.
  Use this to separate concerns: real-time processing + archival + analytics.
```

In [ ]:
# Streaming pipeline design walkthrough

import math

class StreamingPipelineDesign:
    """
    Streaming pipeline design — walks through component selection
    and sizing calculations for a system design interview.
    """

    def __init__(self, name, peak_events_per_min, bytes_per_event, latency_sla_sec):
        self.name = name
        self.peak_eps = peak_events_per_min / 60  # events per second
        self.bytes_per_event = bytes_per_event
        self.latency_sla_sec = latency_sla_sec
        self.mb_per_sec = (self.peak_eps * bytes_per_event) / 1e6

    def kinesis_shards(self):
        # 1 shard: 1 MB/s write, 1000 records/s
        shards_rps = math.ceil(self.peak_eps / 1000)
        shards_bw  = math.ceil(self.mb_per_sec / 1.0)
        return max(shards_rps, shards_bw, 1) + 2  # +2 buffer for spikes

    def consumer_design(self):
        if self.latency_sla_sec <= 5:
            return "Lambda per shard (low latency, no cluster startup)"
        elif self.latency_sla_sec <= 60:
            return "Kinesis Analytics (Flink) or Lambda with tumbling window"
        return "EMR Spark Structured Streaming (complex stateful processing)"

    def dedup_strategy(self):
        return [
            "Client generates UUID event_id per event",
            "Lambda checks DynamoDB for event_id before processing (conditional write)",
            "TTL on DynamoDB dedup table: 24 hours (Kinesis retains 7 days)",
            "bisect_batch_on_function_error=True (isolate poison pills)",
            "Failed events → SQS DLQ → alert on DLQ depth > 0",
        ]

    def archival_design(self):
        return (
            "Kinesis Firehose (separate consumer) → S3 Parquet\n"
            "  Buffer: 128 MB or 60 seconds (whichever first)\n"
            "  Partition: s3://bucket/raw/year=Y/month=M/day=D/hour=H/\n"
            "  Athena table on top for ad-hoc queries"
        )

    def failure_modes(self):
        return [
            ("Kinesis shard hot",    "Increase shard count; use random partition key prefix"),
            ("Lambda throttle",      "Request concurrency increase; use reserved concurrency"),
            ("DynamoDB write limit", "Use on-demand mode; or request provisioned WCU increase"),
            ("Poison pill event",    "bisect batch → single event → DLQ → alert + inspect"),
            ("Consumer lag grows",   "Add more shards OR increase Lambda batch size"),
        ]

    def print_design(self):
        print(f"STREAMING PIPELINE DESIGN: {self.name}")
        print("=" * 60)
        print(f"  Peak events/sec:  {self.peak_eps:,.0f}")
        print(f"  MB/sec:           {self.mb_per_sec:.2f}")
        print(f"  Latency SLA:      <{self.latency_sla_sec}s")
        print(f"  Kinesis shards:   {self.kinesis_shards()}")
        print(f"\n[CONSUMER]   {self.consumer_design()}")
        print(f"\n[ARCHIVAL]\n{self.archival_design()}")
        print("\n[DEDUP / EXACTLY-ONCE]")
        for s in self.dedup_strategy():
            print(f"  - {s}")
        print("\n[FAILURE MODES]")
        for mode, response in self.failure_modes():
            print(f"  {mode:<26} → {response}")


design = StreamingPipelineDesign(
    name="Clickstream Fraud Detection",
    peak_events_per_min=500_000,
    bytes_per_event=512,
    latency_sla_sec=30
)
design.print_design()

<a id='7'></a>
## 7. Pattern 3: Lambda / Kappa Architecture

---

```
LAMBDA ARCHITECTURE (batch + speed layer)
──────────────────────────────────────────
  Source → [Batch layer: Spark on all history] → [Serving layer]
         → [Speed layer: streaming, recent data] → [Serving layer]

  Pros:  batch layer is accurate (reprocesses history); speed is fresh
  Cons:  two codebases, two pipelines — consistency is hard to maintain
         ("which layer is right if they disagree?")

  Use when:
    - Historical accuracy is critical (e.g., financial reporting)
    - Real-time approximation acceptable for operational dashboards
    - Team has capacity to maintain two systems

KAPPA ARCHITECTURE (stream-only)
──────────────────────────────────
  Source → [Stream processing: Kafka/Kinesis + Flink/Spark]
         → [Replayable stream as the source of truth]

  Reprocessing: replay the stream from the beginning with new code

  Pros:  one codebase; stream is the source of truth; simpler
  Cons:  reprocessing takes time; stream retention costs money
         not every stream supports arbitrary replay (Kinesis: 7 days)

  Use when:
    - Team prefers simplicity over marginal accuracy gain
    - Stream can be retained long enough for reprocessing
    - Streaming framework handles late data well (watermarks)

MODERN LAKEHOUSE (emerging standard)
──────────────────────────────────────
  Source → Kinesis/Kafka
         → Flink/Spark Structured Streaming
         → Delta Lake / Iceberg on S3
            (ACID, time travel, schema evolution)
         → Athena / Redshift Spectrum / Databricks SQL

  Combines: streaming ingest + ACID storage + SQL serving
  This is the answer most interviewers at modern data companies want.
```

In [ ]:
# Architecture selector — recommend Lambda, Kappa, or Lakehouse

def recommend_architecture(
    latency_sla_minutes,
    history_reprocessing_needed,
    team_size,
    stream_retention_days,
    exactly_once_required,
):
    """
    Recommend an architecture pattern given requirements.
    The reasoning is as important as the recommendation in interviews.
    """
    print("ARCHITECTURE RECOMMENDATION")
    print("=" * 50)
    print(f"  Latency SLA:            {latency_sla_minutes} min")
    print(f"  History reprocessing:   {history_reprocessing_needed}")
    print(f"  Team size:              {team_size}")
    print(f"  Stream retention:       {stream_retention_days} days")
    print(f"  Exactly-once required:  {exactly_once_required}")

    # Decision logic with reasoning
    print("\nREASONING:")

    if latency_sla_minutes > 60 and not history_reprocessing_needed:
        print("  Latency is batch-tolerable and no reprocessing needed.")
        rec = "Batch-only (Glue + Airflow). Simplest, cheapest."
    elif team_size < 5 and latency_sla_minutes < 60:
        print("  Small team — avoid two-codebase Lambda overhead.")
        if stream_retention_days >= 30:
            print("  Stream retention sufficient for reprocessing.")
            rec = "Kappa Architecture (Kinesis/Kafka + Flink/Spark). One codebase."
        else:
            print("  Stream retention too short for reprocessing → need S3 backup.")
            rec = "Lakehouse (Kinesis → Spark → Delta Lake on S3). Modern default."
    elif history_reprocessing_needed and exactly_once_required:
        print("  Historical accuracy + exactly-once → Lambda safest.")
        rec = "Lambda Architecture (Spark batch for history, streaming for real-time)."
    else:
        print("  Modern stack available → Lakehouse preferred.")
        rec = "Lakehouse (Kinesis → Spark Structured Streaming → Delta Lake / Iceberg)."

    print(f"\nRECOMMENDATION: {rec}")

    print("\nTRADE-OFFS GIVEN UP:")
    if "Lakehouse" in rec:
        tradeoffs = [
            "Flink/Spark cluster startup adds ~2-5 min latency vs Lambda",
            "Delta Lake ACID transactions slower than raw S3 for very high write throughput",
            "Schema evolution requires careful management (forwards/backwards compat)",
        ]
    elif "Kappa" in rec:
        tradeoffs = [
            "Reprocessing takes as long as stream replay time (could be hours)",
            "Must keep stream hot for extended retention (cost at Kinesis scale)",
            "Late data handling requires careful watermark tuning",
        ]
    elif "Lambda" in rec:
        tradeoffs = [
            "Two codebases — must keep batch and streaming logic in sync",
            "Merging batch + speed layer results adds complexity",
            "Batch layer has high latency (full recompute)",
        ]
    else:
        tradeoffs = ["No real-time capability — latency bounded by batch schedule"]

    for t in tradeoffs:
        print(f"  - {t}")


print("SCENARIO A: Large fintech, strict audit requirements")
recommend_architecture(
    latency_sla_minutes=5,
    history_reprocessing_needed=True,
    team_size=20,
    stream_retention_days=7,
    exactly_once_required=True,
)

print("\n" + "─"*55)
print("SCENARIO B: SaaS startup, 3-person data team")
recommend_architecture(
    latency_sla_minutes=10,
    history_reprocessing_needed=False,
    team_size=3,
    stream_retention_days=30,
    exactly_once_required=False,
)

<a id='8'></a>
## 8. Pattern 4: AWS Stack — End-to-End

---

```
END-TO-END AWS DATA PIPELINE (the default modern answer)
──────────────────────────────────────────────────────────────────

  SOURCES
  ────────
  Operational DB (RDS/Aurora)     → DMS → S3 (CDC)
  Application events              → Kinesis Data Streams
  Files / uploads                 → S3 PUT event → Lambda trigger
  Third-party API                 → Lambda (scheduled pull)

  INGEST
  ───────
  Streaming:  Kinesis Data Streams (ordered, replayable, fan-out)
  Batch CDC:  AWS DMS (minimal DB impact, CDC log reading)
  File:       S3 event notification → SQS → Lambda

  PROCESS
  ────────
  Low volume (<10 GB/day):   AWS Glue (serverless Spark, pay-per-use)
  Medium volume (10-1TB/day): EMR Spark (more control, spot instances)
  Stream processing:         Kinesis Data Analytics (Flink) or Lambda
  ETL transformations:       dbt on Redshift (SQL-based, version controlled)

  STORE
  ──────
  Raw:       S3 (Parquet, partitioned by date)
  Curated:   S3 + Delta Lake / Iceberg (ACID, time travel)
  Aggregate: Redshift (BI queries, complex SQL)
  Operational: DynamoDB (<10ms lookups), ElastiCache (caching)

  SERVE
  ──────
  Ad-hoc SQL:    Athena (on S3, pay-per-query, $5/TB)
  BI dashboards: Redshift + QuickSight / Tableau
  API serving:   Lambda + DynamoDB or ElastiCache
  ML features:   SageMaker Feature Store

  ORCHESTRATE
  ────────────
  Batch DAGs:     MWAA (managed Airflow) or Step Functions
  Scheduled jobs: EventBridge + Lambda
  Monitoring:     CloudWatch + SNS alarms + Datadog
  Data quality:   Great Expectations / dbt tests / custom Lambda
```

In [ ]:
# AWS stack selector — given requirements, print the recommended AWS services

def aws_stack_recommendation(requirements):
    """
    Map requirements to AWS service choices with justification.
    """
    stack = {}

    # Ingest
    if requirements.get("source_type") == "events":
        eps = requirements.get("events_per_sec", 0)
        if eps < 100:
            stack["Ingest"] = "SQS (low volume, simple, managed)"
        elif eps < 10_000:
            shards = max(1, eps // 1000 + 1)
            stack["Ingest"] = f"Kinesis Data Streams ({shards} shards)"
        else:
            stack["Ingest"] = "MSK (Kafka) — Kinesis limit reached at 10k+ events/s per shard"
    elif requirements.get("source_type") == "database":
        stack["Ingest"] = "AWS DMS (CDC mode) → S3 raw zone"
    else:
        stack["Ingest"] = "S3 PUT + SQS notification → Lambda trigger"

    # Process
    gb_per_day = requirements.get("gb_per_day", 1)
    latency_min = requirements.get("latency_minutes", 60)
    if latency_min < 5:
        stack["Process"] = "Lambda (streaming, per-event or micro-batch)"
    elif gb_per_day < 10:
        stack["Process"] = "AWS Glue (serverless Spark, good for <10 GB/day)"
    elif gb_per_day < 1000:
        stack["Process"] = "EMR Spark (managed cluster, spot instances for cost)"
    else:
        stack["Process"] = "EMR Spark with autoscaling (+ Spot for non-critical steps)"

    # Store
    stack["Store: Raw"] = "S3 + Parquet + partitioned by event_date"
    if requirements.get("acid_required"):
        stack["Store: Curated"] = "S3 + Delta Lake (ACID, time travel, schema evolution)"
    else:
        stack["Store: Curated"] = "S3 + Parquet (simpler, Athena-queryable)"
    if requirements.get("bi_dashboards"):
        stack["Store: DW"] = "Redshift (distkey on join col, sortkey on date)"
    if requirements.get("key_value_lookups"):
        stack["Store: OLTP"] = "DynamoDB (single-table design, on-demand capacity)"

    # Serve
    stack["Serve: SQL"] = "Athena (ad-hoc, $5/TB) + Redshift (BI, concurrent)"
    stack["Orchestrate"] = "MWAA (Airflow) for DAGs; EventBridge for cron"
    stack["Monitor"] = "CloudWatch alarms + SNS + Datadog for business metrics"

    print("AWS STACK RECOMMENDATION")
    print("=" * 55)
    for layer, service in stack.items():
        print(f"  {layer:<22} → {service}")


print("SCENARIO: E-commerce clickstream + orders pipeline")
aws_stack_recommendation({
    "source_type": "events",
    "events_per_sec": 5_000,
    "gb_per_day": 250,
    "latency_minutes": 10,
    "acid_required": True,
    "bi_dashboards": True,
    "key_value_lookups": True,
})

print("\n" + "─"*55)
print("SCENARIO: Small SaaS DB replica for analysts")
aws_stack_recommendation({
    "source_type": "database",
    "gb_per_day": 5,
    "latency_minutes": 120,
    "acid_required": False,
    "bi_dashboards": True,
    "key_value_lookups": False,
})

<a id='9'></a>
## 9. Pattern 5: Trade-off Reasoning & SLAs

---

```
WHAT INTERVIEWERS WANT:
  Not the "correct" answer. The "reasoned" answer.
  "I chose X because of constraints Y and Z. This trades off A for B.
   If the constraint changed, I'd switch to C."

SLA MATH (know these cold):
  99.0%  = 3.65 days/year downtime    = 7.2 hours/month
  99.5%  = 1.83 days/year downtime    = 3.6 hours/month
  99.9%  = 8.76 hours/year downtime   = 43.8 min/month
  99.95% = 4.38 hours/year downtime   = 21.9 min/month
  99.99% = 52.6 min/year downtime     = 4.4 min/month

COMMON TRADE-OFF PAIRS (be ready to defend either side):

  Exactly-once vs At-least-once
    Exactly-once: Kafka transactions, higher latency, complex
    At-least-once: simpler, faster — acceptable with idempotent consumers
    Choose: at-least-once + idempotent write = effectively exactly-once at lower cost

  Consistency vs Availability (CAP theorem)
    Consistent: all nodes see same data — bank balance must be accurate
    Available:  always responds — clickstream ok if slightly stale
    Choose based on: what breaks if stale? (user-visible money = consistent)

  Schema-on-write vs Schema-on-read
    Write: reject bad data at ingest — clean lake; rigid
    Read:  accept all data — flexible; garbage can accumulate
    Choose: schema-on-write for regulated data; schema-on-read for exploration

  Managed service vs DIY
    Managed (Glue, Kinesis, MWAA): less ops, higher cost per unit
    DIY (EMR, self-hosted Kafka, Apache Airflow): more control, more ops
    Choose: managed for small teams or early stages; DIY at high scale

SLOW MOTION TRACE — Defending Kinesis over Kafka:
  Interviewer: "Why not Kafka? It's more powerful."
  You: "Kafka is more powerful — it has infinite retention,
        better ecosystem, lower cost at high throughput.
        I chose Kinesis because:
        1. Fully managed — our 3-person team can't run Kafka brokers.
        2. Native IAM integration — no Zookeeper, no TLS config.
        3. Our volume (8k events/sec) fits Kinesis budget.
        If we hit 50k events/sec or need >7-day retention, I'd migrate to MSK."
```

In [ ]:
# Trade-off reasoning framework — practice defending any technical choice

class TechnicalDecision:
    """
    Structure a technical choice for interview delivery.
    Format: chose X over Y because Z; trade-off is A; switch to B if C changes.
    """

    def __init__(self, choice, alternatives, reasons, tradeoffs, switch_when):
        self.choice = choice
        self.alternatives = alternatives  # list of (name, why_rejected)
        self.reasons = reasons
        self.tradeoffs = tradeoffs
        self.switch_when = switch_when

    def deliver(self):
        print(f"DECISION: {self.choice}")
        print("─" * 55)
        print("\nWHY THIS:")
        for r in self.reasons:
            print(f"  + {r}")
        print("\nALTERNATIVES CONSIDERED:")
        for alt, rejection in self.alternatives:
            print(f"  - {alt}: rejected because {rejection}")
        print("\nWHAT WE GIVE UP:")
        for t in self.tradeoffs:
            print(f"  ~ {t}")
        print("\nWHEN I'D SWITCH:")
        for sw in self.switch_when:
            print(f"  → {sw}")


# Practice: Kinesis vs Kafka decision
kinesis_decision = TechnicalDecision(
    choice="Kinesis Data Streams (over Kafka / MSK)",
    alternatives=[
        ("MSK (Kafka)",  "operational overhead too high for 3-person team; Zookeeper, broker sizing, security config"),
        ("SQS",          "doesn't preserve ordering per partition key — financial events need ordering"),
        ("EventBridge",  "500 event/sec limit — too low at 8k events/sec peak"),
    ],
    reasons=[
        "Fully managed: no broker ops, patching, or Zookeeper",
        "Native IAM: no TLS cert management or ACL config",
        "Per-shard ordering: customer events arrive in order within shard",
        "Fan-out: fraud detection + archival + analytics as independent consumers",
        "8k events/sec fits within 9 shards comfortably",
    ],
    tradeoffs=[
        "7-day max retention (vs Kafka: unlimited) — must archive to S3 for replay",
        "$0.015/shard-hour cost — at 1000 shards, MSK is cheaper",
        "Fewer ecosystem connectors than Kafka",
        "Shard count must be set manually (vs Kafka auto-partition)",
    ],
    switch_when=[
        "Volume exceeds 50k events/sec (Kinesis cost becomes prohibitive)",
        "Need >7-day stream replay for reprocessing",
        "Team grows: dedicated platform team to manage Kafka ops",
    ]
)

kinesis_decision.deliver()

# SLA math helper
print("\n\nSLA DOWNTIME CALCULATOR")
print("─" * 40)
for uptime in [99.0, 99.5, 99.9, 99.95, 99.99]:
    downtime_min_month = (1 - uptime/100) * 30 * 24 * 60
    downtime_hr_year   = (1 - uptime/100) * 365 * 24
    print(f"  {uptime:.2f}%  →  {downtime_min_month:>6.1f} min/month  |  {downtime_hr_year:.2f} hr/year")

<a id='10'></a>
## 10. Full Decision Map

```
SYSTEM DESIGN INTERVIEW DECISION MAP
──────────────────────────────────────────────────────────────────
REQUIREMENT                     CHOICE               ALTERNATIVE IF X
──────────────────────────────────────────────────────────────────
Latency < 1s                    Lambda + DynamoDB    Flink if stateful
Latency 1-5 min                 Kinesis + Lambda     Glue micro-batch
Latency hourly                  Glue + Airflow       EMR if large
Latency daily                   Glue + S3 batch      dbt + Redshift

Volume < 100 events/s           SQS + Lambda         —
Volume 100-10k events/s         Kinesis              —
Volume > 10k events/s           MSK (Kafka)          Kinesis (if managed ok)

Query: key lookup               DynamoDB             ElastiCache (cache)
Query: ad-hoc SQL on S3         Athena               Presto/Trino self-hosted
Query: BI dashboards            Redshift             BigQuery / Snowflake
Query: full text search         OpenSearch           —

Exactly-once required           Kafka txns + Spark   Idempotent + at-least-once
At-least-once ok                Kinesis + Lambda     SQS standard

ACID on data lake               Delta Lake           Iceberg
Schema enforcement              Glue Schema Registry Confluent Schema Registry
Schema evolution                Iceberg              Delta Lake (also good)

CDC from OLTP DB                AWS DMS              Debezium (Kafka)
Batch extract                   JDBC + Glue          Fivetran (managed)

Orchestrate batch               MWAA (Airflow)       Step Functions
Orchestrate streaming           None needed          Step Functions + Lambda

Monitor pipelines               CloudWatch + Alarms  Datadog
Data quality                    Great Expectations   dbt tests + custom Lambda
```

<a id='11'></a>
## 11. Interview Cheat Sheet

### When to use each pattern:

| Signal | Pattern |
|--------|--------|
| "Design a data pipeline" | Start with batch, add streaming if latency requires |
| "Real-time analytics" | Kinesis → Lambda/Flink → DynamoDB → serving |
| "100x more data" | Identify bottleneck; shard/partition/scale that layer |
| "Historical accuracy" | Lambda arch or Lakehouse with reprocessing |
| "Small team, simple" | Kappa or Lakehouse; avoid two-codebase Lambda |
| "Exactly-once" | Idempotent writes + dedup key (cheaper than Kafka txns) |

### Back-of-envelope — memorize these:

```
1M events/day  = ~12 events/sec
1B events/day  = ~12k events/sec
1KB × 1M/day   = 1 GB/day raw;  ~140 MB/day Parquet
1KB × 1B/day   = 1 TB/day raw;  ~140 GB/day Parquet
Kinesis shard: 1 MB/s write, 1000 records/s, $0.015/shard-hour
S3: ~$0.023/GB/month
Athena: $5/TB scanned (Parquet = ~25,000x cheaper than raw JSON)
DynamoDB: 1 WCU = 1 write/s up to 1KB
Lambda: max 15 min, 10GB RAM, 1000 concurrent (soft)
```

### Common templates:

```
# BATCH PIPELINE TEMPLATE
Source → DMS/JDBC → S3 raw (Parquet, partitioned by date)
  → Glue Spark (clean, dedupe, transform)
  → S3 curated (Delta Lake)
  → Redshift COPY (for BI)
  Orchestration: Airflow DAG, max_active_runs=1, DQ gate before load

# STREAMING PIPELINE TEMPLATE
Source → Kinesis (N shards) → Lambda (fraud / alerts)
                            → Firehose → S3 (archival)
  DynamoDB: dedup + state; SNS: alerts
  Athena on S3: historical analysis

# LAKEHOUSE TEMPLATE
Source → Kinesis → Spark Structured Streaming
  → Delta Lake on S3 (ACID, time travel, schema evolution)
  → Athena / Redshift Spectrum (SQL serving)
  → MWAA DAGs for maintenance (VACUUM, OPTIMIZE)

# TRADE-OFF TEMPLATE (say this every time you make a choice)
"I chose X over Y because [constraint]. This trades off [A] for [B].
 If [condition changed], I'd switch to [Z]."
```

### Gotchas:

```
❌  Drawing before asking about volume and latency requirements
❌  Choosing Kafka for every streaming problem (overkill at small scale)
❌  Forgetting idempotency — every step must be safe to retry
❌  Not mentioning failure modes and monitoring
❌  Exact numbers without back-of-envelope ("I'd need to benchmark")
✅  Always size Kinesis shards: events/sec ÷ 1000 = shard count
✅  Always mention partition strategy and why (query pattern driven)
✅  Always name what you're giving up when making a choice
✅  End with: "The most important thing to get right here is X"
```

<a id='12'></a>
## 12. Summary Map

```
SYSTEM DESIGN WHITEBOARD
═══════════════════════════════════════════════════════════════

  FRAMEWORK: CLARIFY → SKETCH → DEEP-DIVE → SCALE → FAIL → TRADE-OFF

  DATA PIPELINE ARCHETYPES
  ─────────────────────────────────────────────────────────────
  Batch
    Source → DMS/JDBC → S3 raw → Glue → S3 curated → Redshift
    Key: idempotency, DQ gate, Airflow DAG, partition by event_date

  Streaming
    Source → Kinesis → Lambda → DynamoDB (state) + SNS (alerts)
           → Firehose → S3 (archive) → Athena (analysis)
    Key: shard sizing, dedup key, DLQ, bisect on failure

  Lakehouse (modern default)
    Source → Kinesis → Spark Structured Streaming
           → Delta Lake / Iceberg on S3
           → Athena / Redshift Spectrum
    Key: ACID, time travel, schema evolution, one codebase

  SIZING NUMBERS
  ─────────────────────────────────────────────────────────────
  1M events/day  → 12 eps  → 1 Kinesis shard
  100M events/day → 1.2k eps → 2 Kinesis shards
  1B events/day  → 12k eps → 13 Kinesis shards → consider MSK
  1KB/event × 1B/day = 1 TB raw → 140 GB Parquet → $3.2/month S3

  KEY TRADE-OFF PAIRS
  ─────────────────────────────────────────────────────────────
  Exactly-once vs At-least-once: idempotent write = effectively same
  Managed vs DIY:   managed early; DIY at >$50k/month
  Batch vs Stream:  batch is simpler; stream when latency < 5 min
  Schema-on-write vs -read: write for regulated; read for exploration

  INTERVIEW DELIVERY
  ─────────────────────────────────────────────────────────────
  Min 3 clarifying questions before drawing
  Name the trade-off every time you make a choice
  Back-of-envelope: size every component numerically
  End with: "The hardest part of this design is X"

────────────────────────────────────────────────────────────────
*End of System Design Whiteboard Master Guide — Sean Edition*
```